In [ ]:
from pathlib import Path
import sys

import anndata as ad
import pandas as pd
import torch


def find_project_root() -> Path:
    """Resolve the repository root when running from the root or analyses directory."""
    current_dir = Path.cwd().resolve()
    for candidate in (current_dir, current_dir.parent):
        if (candidate / "src").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from the project root or the analyses directory.")


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data" / "01_single_cell_prediction"
RESULTS_DIR = PROJECT_ROOT / "results" / "01_single_cell_prediction"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from scvolt.model import SCVOLTModel
from scvolt import tools

TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
latent_dim = 30
num_DRGs = 1200
sc_file_name = "451Lu_PLX-4720_anndata.h5ad"
drug_name = "PLX-4720"


In [ ]:
"""Run scVOLT encoding and prediction for one dataset/drug configuration."""
tools.set_model_seed(43)
# Load scRNA-seq data.
sc_expr = ad.read_h5ad(DATA_DIR / sc_file_name)
labels = sc_expr.obs["response"]
sc_expr = pd.DataFrame(data=sc_expr.X, index=sc_expr.obs.index, columns=sc_expr.var.index)
data = {"sc": sc_expr}
# Define the model.
model = SCVOLTModel(latent_dim=latent_dim, drug_name=drug_name, num_DRGs=num_DRGs)
# Perform the training.
encoded_data = model.encode(data=data, device=TORCH_DEVICE)
sc_labels = labels[labels.index.isin(sc_expr.index)]
pred = model.predict(encoded_data=encoded_data, control_fdr=False)
# y_true contains 0/1 labels; y_pred contains model-predicted scores.
metrics = tools.get_metrics(1 - sc_labels, pred["sc_scores"])
print("AUROC:", metrics["AUROC"])
print("AUPRC:", metrics["AUPRC"])
print("Cohens_d:", metrics["Cohens_d"])


In [ ]:
saved_data = pd.DataFrame(
    [[metrics["AUROC"], metrics["AUPRC"], metrics["Cohens_d"]]],
    columns=["AUROC", "AUPRC", "Cohens_d"],
)
output_file = RESULTS_DIR / f"{sc_file_name}_k_{latent_dim}_DRGs_{num_DRGs}.csv"
saved_data.to_csv(output_file, header=True)


In [ ]:
latent_dim = 20
num_DRGs = 1200
sc_file_name = "Docetaxel_data.h5ad"
drug_name = "Docetaxel"


In [ ]:
"""Run scVOLT encoding and prediction for one dataset/drug configuration."""
tools.set_model_seed(43)
# Load scRNA-seq data.
sc_expr = ad.read_h5ad(DATA_DIR / sc_file_name)
labels = sc_expr.obs["response"]
sc_expr = pd.DataFrame(data=sc_expr.X, index=sc_expr.obs.index, columns=sc_expr.var.index)
data = {"sc": sc_expr}
# Define the model.
model = SCVOLTModel(latent_dim=latent_dim, drug_name=drug_name, num_DRGs=num_DRGs)
# Perform the training.
encoded_data = model.encode(data=data, device=TORCH_DEVICE)
sc_labels = labels[labels.index.isin(sc_expr.index)]
pred = model.predict(encoded_data=encoded_data, control_fdr=False)
# y_true contains 0/1 labels; y_pred contains model-predicted scores.
metrics = tools.get_metrics(1 - sc_labels, pred["sc_scores"])
print("AUROC:", metrics["AUROC"])
print("AUPRC:", metrics["AUPRC"])
print("Cohens_d:", metrics["Cohens_d"])


In [ ]:
saved_data = pd.DataFrame(
    [[metrics["AUROC"], metrics["AUPRC"], metrics["Cohens_d"]]],
    columns=["AUROC", "AUPRC", "Cohens_d"],
)
output_file = RESULTS_DIR / f"{sc_file_name}_k_{latent_dim}_DRGs_{num_DRGs}.csv"
saved_data.to_csv(output_file, header=True)
